<a href="https://colab.research.google.com/github/shoh0806/Deep-Learning-Project/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 라이브러리

In [7]:
!pip install transformers -q  #이게 HuggingFace 설치 하는겁니당!

# 2. 라이브러리 import

In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True  # 이건 GPU 연산을 항상 같은 순서로 실행하게 하는 코드입니다 없으면 gpu에서 랜덤성이 생길 수 있다고 합니다.
    torch.backends.cudnn.benchmark = False  #GPU 최적화 알고리즘 고정 , 없으면 실행할 때마다 다른 알고리즘 선택할 수 있음

set_seed(42)
# 필요한것들 import하고 seed 42로 고정했습니다!

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cuda


# 3. Drive 연결 및 데이터 로드

In [9]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/clickbait_project'
CSV_PATH = f'{BASE_DIR}/dataset.csv'

df = pd.read_csv(CSV_PATH)
print(f"전체 데이터: {len(df)}개")
print(f"클릭베이트(1): {len(df[df['label']==1])}개")
print(f"정상(0): {len(df[df['label']==0])}개")
print(df.head())

#저 같은 경우에는 구글드라이브에 폴더 만들고 거기안에 thumbnails폴더랑 csv 파일을 넣고 thumnails폴더안에 img를 넣어놨습니다 그래서
# 거기있는걸 읽어서 사용하려고 이렇게 짰는데 만약에 다른 방식으로 로컬로 짜실거면 그렇게 바꾸셔도 되고요 아니면 저처럼 구글 드라이브에 데이터들 넣고 마운트해서 쓰셔도됩니다
# 어떤구조로 만들었는지 똑같이 만들어서 돌려보고싶으신거면 저한테 편하게 물어봐 주세용

Mounted at /content/drive
전체 데이터: 2512개
클릭베이트(1): 1110개
정상(0): 1402개
      video_id                                              title  \
0  lepHJI4ze-w               I Bought 250 BANNED Amazon Products!   
1  ezwXAlJ5pAY  TikTok Wigofellas PRANKS on MOM - Wigofellas P...   
2  i6_OomloWUg  TikTok Wigofellas PRANKS on MOM - Wigofellas P...   
3  VvuuR7HifUY        Superheroes in Real Life Caught On Camera !   
4  uwIjrEbywnk  20 People You Won't Believe Existed Till You S...   

               thumbnail_path  label  
0  thumbnails/lepHJI4ze-w.jpg      1  
1  thumbnails/ezwXAlJ5pAY.jpg      1  
2  thumbnails/i6_OomloWUg.jpg      1  
3  thumbnails/VvuuR7HifUY.jpg      1  
4  thumbnails/uwIjrEbywnk.jpg      1  


# 4. 데이터 분할

In [10]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"train: {len(train_df)}개")
print(f"val:   {len(val_df)}개")
print(f"test:  {len(test_df)}개")

# train_test validation은 8: 1:1 로했고 성능에대해서는 llm물어봤는데 우리처럼 2500개 정도면 적은편이니 8:1:1 이 맞다고합니다
# 7:1.5:1.5 나 6:2:2보단
# 그리고
# 저희는 비교하는게 목적이니까 random_state랑 seed는 고정해서 진행하는게 좋다고생각합니다 그래서 종호님같은경우 cross at 짜주실때 randomstate는 42로 ssed도 42로 고정해서 진행해주시면
# 감사하겠습니다!

train: 2009개
val:   251개
test:  252개


# 5. Dataset 클래스

In [11]:
# tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')  #(mbert 쓸 경우 이 코드로 변경)
tokenizer = BertTokenizer.from_pretrained('bert-base-cased') #(bert-cased)
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['title'].tolist() #타이틀을 리스트화 진행했고요
        self.labels = df['label'].tolist() # 정답,오답 인 label을 list화한겁니다.
        self.tokenizer = tokenizer
        self.max_len = max_len #성능에는 영향이 크게 없지만 click bait특성상 제목(title) 이고 그래서 문장이 짧다보니까 max_len을 현재는 128로 돌렸는데 64로 해도 될거같습니다.
        #아래 코드에서 확인한 결과 평균이 20개 95%가 50개 이하면 max_len을 64로 줄여도 된다고 합니다 그러면 돌리는데 속도는 더 줄어들겁니다 max_len 몇으로 할지 같은 하이퍼파라미터는 다음
        #회의때 같이 의논하면 좋을거같습니다.

    def __len__(self):
        return len(self.texts) #리스트 안에 제목이 몇개인지

    def __getitem__(self, idx): #idx번째 데이터를 토크나이징해서 반환
        tokens = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      tokens['input_ids'].squeeze(),
            'attention_mask': tokens['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextDataset(train_df, tokenizer)
val_dataset   = TextDataset(val_df, tokenizer)
test_dataset  = TextDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Dataset 준비 완료!")
print(f"train loader: {len(train_loader)}배치")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset 준비 완료!
train loader: 63배치


#5.5 데이터 확인 코드

In [12]:
import matplotlib.pyplot as plt

lengths = [len(tokenizer.encode(title)) for title in df['title']]
print(f"평균: {np.mean(lengths):.1f}")
print(f"최대: {max(lengths)}")
print(f"95%: {np.percentile(lengths, 95):.1f}")
print(f"99%: {np.percentile(lengths, 99):.1f}")

평균: 20.2
최대: 71
95%: 36.0
99%: 45.9


# 6.  BERT 모델 정의

In [13]:
class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # self.bert = BertModel.from_pretrained('bert-base-multilingual-cased') (mbert 쓸 경우 이 코드로 변경)
        self.bert = BertModel.from_pretrained('bert-base-cased') #( bert-cased쓸 경우 )
        # self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):# input_ids는 토큰 번호들이고  attention_mask는 패딩 마스크입니다 (32,128)
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_vector = outputs.last_hidden_state[:, 0, :] #outputs.last_hidden_state shape: (32, 128, 768) 배치 토큰수 차원이니까 [:, 0 , :] 는 전체 배치 0번쨰 토큰 전체차원
        #batch first없는데 RNN이랑 다르게 BERT는 처음부터 배치가 첫 번째 라고합니다 그래서 안써도 이렇게 나오는겁니다 혹시 batch_first없었는데?... 라고 코드보다 오해하실까봐 미리
        #알려드림.
        #즉, 32개 문장 각각의 [CLS] 벡터
        # [CLS]는 입력시엔 빈 특수토큰이지만
        # BERT 12개 통과 후엔 문장 전체 의미를 담은 벡터가 됨
        cls_vector = self.dropout(cls_vector)
        return self.classifier(cls_vector)

model = BertClassifier().to(device)
print("BERT 모델 준비 완료!")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 아 positioning encoding은 bert같은경우는 이미 내부에서 자동처리하니까 필요없습니다!

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT 모델 준비 완료!
파라미터 수: 108,311,810


#7. 학습함수

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds, targets = 0, [], [] # 전체 loss 합산용 (나중에 평균 냄) , pred는 모델 예측값 저장리스트 , target은 실제 정답 저장 리스트입니당

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds   += outputs.argmax(1).cpu().tolist()
        targets += labels.cpu().tolist()

    return total_loss/len(loader), f1_score(targets, preds, average='macro'), accuracy_score(targets, preds)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss    = criterion(outputs, labels)

            total_loss += loss.item()
            preds   += outputs.argmax(1).cpu().tolist()
            targets += labels.cpu().tolist()

    return total_loss/len(loader), f1_score(targets, preds, average='macro'), accuracy_score(targets, preds)

print("학습 함수 준비 완료!")



학습 함수 준비 완료!


# 8. 학습실행

In [15]:
EPOCHS = 10
best_val_f1 = 0
patience_counter = 0
PATIENCE = 3

for epoch in range(EPOCHS):
    train_loss, train_f1, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_acc       = eval_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  train loss:{train_loss:.4f} f1:{train_f1:.4f} acc:{train_acc:.4f}")
    print(f"  val   loss:{val_loss:.4f} f1:{val_f1:.4f} acc:{val_acc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        # torch.save(model.state_dict(), f'{BASE_DIR}/mbert_best.pt') (mbert 쓸 경우 이 코드로 변경)
        torch.save(model.state_dict(), f'{BASE_DIR}/bert_cased_best.pt')
        # torch.save(model.state_dict(), f'{BASE_DIR}/bert_uncased_best.pt')
        print(f"  → 모델 저장!")
    else:
        patience_counter += 1
        print(f"  → patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print(f"\nEarly Stopping! 최고 val f1: {best_val_f1:.4f}")
            break

Epoch 1/10
  train loss:0.4504 f1:0.7755 acc:0.7785
  val   loss:0.3593 f1:0.8463 acc:0.8486
  → 모델 저장!
Epoch 2/10
  train loss:0.2347 f1:0.9002 acc:0.9014
  val   loss:0.4244 f1:0.8565 acc:0.8566
  → 모델 저장!
Epoch 3/10
  train loss:0.1192 f1:0.9566 acc:0.9572
  val   loss:0.4404 f1:0.8590 acc:0.8606
  → 모델 저장!
Epoch 4/10
  train loss:0.0514 f1:0.9824 acc:0.9826
  val   loss:0.4455 f1:0.8718 acc:0.8725
  → 모델 저장!
Epoch 5/10
  train loss:0.0232 f1:0.9899 acc:0.9900
  val   loss:0.5577 f1:0.8762 acc:0.8765
  → 모델 저장!
Epoch 6/10
  train loss:0.0245 f1:0.9904 acc:0.9905
  val   loss:0.7689 f1:0.8725 acc:0.8725
  → patience 1/3
Epoch 7/10
  train loss:0.0296 f1:0.9884 acc:0.9886
  val   loss:0.5554 f1:0.8644 acc:0.8645
  → patience 2/3
Epoch 8/10
  train loss:0.0129 f1:0.9955 acc:0.9955
  val   loss:0.6993 f1:0.8566 acc:0.8566
  → patience 3/3

Early Stopping! 최고 val f1: 0.8762


# 9. 테스트 평가

In [16]:
# 최고 모델 불러와서 test 평가
# model.load_state_dict(torch.load(f'{BASE_DIR}/mbert_best.pt')) (mbert 쓸 경우 이 코드로 변경)
model.load_state_dict(torch.load(f'{BASE_DIR}/bert_cased_best.pt')) #( bert-cased쓸 경우 )
# model.load_state_dict(torch.load(f'{BASE_DIR}/bert_uncased_best.pt'))
test_loss, test_f1, test_acc = eval_epoch(model, test_loader, criterion)

print("========== BERT 최종 결과 ==========")
print(f"test F1:  {test_f1:.4f}")
print(f"test Acc: {test_acc:.4f}")

# 나중에 비교용으로 저장
bert_result = {'f1': test_f1, 'acc': test_acc}

========== BERT 최종 결과 ==========
test F1:  0.8670
test Acc: 0.8690
